# Assignment 9 – SVM, Naive Bayes, Decision Tree & Bagging

---
## Question 1: Hyperplane, Support Vectors, Margin, and Maximum Margin Classifier

### Hyperplane
A hyperplane is a decision boundary that separates data points belonging to different classes. In an n-dimensional feature space, the hyperplane is an (n-1)-dimensional subspace. For 2D data it is a line; for 3D it is a plane; for higher dimensions it is called a hyperplane.

**Equation: `w · x + b = 0`**

where `w` is the weight vector (normal to the hyperplane), `x` is the input feature vector, and `b` is the bias (intercept). Points where `w·x + b > 0` are classified as one class, and `w·x + b < 0` as the other class.

### Support Vectors
Support vectors are the data points from each class that lie **closest** to the decision boundary (hyperplane). They are the most critical, hardest-to-classify points — they 'support' or define the position and orientation of the hyperplane. If these points were removed, the hyperplane would change.

These are the only points that determine the SVM's decision boundary — all other training points play no role once training is complete, making SVM memory-efficient in deployment.

### Margin
The margin is the total perpendicular distance between the two parallel margin hyperplanes (one touching the nearest positive-class support vector, one touching the nearest negative-class support vector).

**Margin = 2 / ||w||**

The two margin hyperplanes are: `w·x + b = +1` (positive) and `w·x + b = -1` (negative).

### Maximum Margin Classifier
The Maximum Margin Classifier (Hard Margin SVM) finds the hyperplane that maximizes the margin:

**Minimize:** `(1/2)||w||²`  
**Subject to:** `y_i(w · x_i + b) >= 1` for all training points i

Maximizing margin is equivalent to minimizing `||w||²`. Only support vectors affect the solution.

### Why Maximizing the Margin Improves Generalization
- Larger margins give the model more tolerance for slight noise in new data.
- VC theory proves that maximizing the margin minimizes an upper bound on generalization error.
- Narrow margins force the hyperplane too close to data points — small perturbations can cause misclassification.


---
## Question 2: Why We Need Kernels in SVM

### The Kernel Trick
Standard (linear) SVM works only when data is linearly separable in the original feature space. Real-world data is often non-linearly distributed. The kernel trick implicitly maps data to a higher-dimensional space where it becomes linearly separable, **without** explicitly computing the coordinates in that higher-dimensional space.

**`K(x_i, x_j) = φ(x_i) · φ(x_j)`**

The kernel function computes the dot product in the high-dimensional space directly from the original features.

### 1. Linear Kernel — `K(x_i, x_j) = x_i · x_j`
- **Intuition:** Simply the dot product — no transformation. Decision boundary remains a hyperplane.
- **Advantages:** Computationally fast, interpretable. Scales well to high-dimensional data.
- **Limitations:** Cannot model non-linear boundaries.
- **Use cases:** Text classification (spam detection), document categorization, genomics.

### 2. Polynomial Kernel — `K(x_i, x_j) = (x_i · x_j + c)^d`
- **Intuition:** Maps data into a polynomial feature space of degree `d`, enabling curved boundaries.
- **Advantages:** Captures non-linear relationships without explicit feature expansion. Flexible.
- **Limitations:** Sensitive to degree `d` and constant `c`. High degrees cause instability and overfitting.
- **Use cases:** Image recognition, handwriting recognition, NLP with moderate non-linearity.

### 3. RBF / Gaussian Kernel — `K(x_i, x_j) = exp(-γ ||x_i - x_j||²)`
- **Intuition:** Measures similarity via Euclidean distance. Implicitly maps to infinite-dimensional space.
- **Advantages:** Extremely flexible — can model any smooth boundary. Most commonly used kernel.
- **Limitations:** Requires tuning C and γ. Slower on large datasets. Less interpretable.
- **Use cases:** Image classification, medical diagnosis, complex non-linear boundaries.


---
## Question 3: Bayes' Theorem and Naive Bayes Classification

### Bayes' Theorem
**`P(C | X) = [P(X | C) × P(C)] / P(X)`**

- **P(C | X) — Posterior:** Probability that a data point belongs to class C given features X. This drives the classification decision.
- **P(X | C) — Likelihood:** Probability of observing features X given class C. Learned from training data.
- **P(C) — Prior:** Probability of class C in the dataset — fraction of training examples in class C.
- **P(X) — Evidence:** Overall probability of features X across all classes. Acts as a normalization constant; ignored in classification since it's the same for all classes.

### How Naive Bayes Uses Bayes' Theorem
**`y_pred = argmax_C [P(C) × P(X₁|C) × P(X₂|C) × ... × P(Xₙ|C)]`**

- **Gaussian NB:** Each `P(Xᵢ|C)` modeled as a Gaussian distribution. For continuous numerical features.
- **Multinomial NB:** `P(Xᵢ|C)` is the frequency of feature i in class C. For text/count data.
- **Bernoulli NB:** Features are binary (present/absent). For text with binary word presence indicators.

### Why It Is Called 'Naive'
It assumes all features are conditionally **independent** given the class label:

**`P(X₁, X₂, ..., Xₙ | C) = P(X₁|C) × P(X₂|C) × ... × P(Xₙ|C)`**

This assumption is almost never true in real data (e.g., 'free' and 'money' co-occur in spam). Yet Naive Bayes works remarkably well in practice because the relative rankings of posterior probabilities are often preserved even when independence is violated.


---
## Question 4: Entropy, Information Gain, and Gini Impurity in Decision Trees

### Entropy
Measures the degree of impurity or disorder in a set of data.

**`Entropy(S) = -Σ p_i × log₂(p_i)`**

- Entropy = 0: All samples belong to one class — perfectly pure leaf.
- Entropy = 1 (binary): 50-50 split — maximum uncertainty.
- Example: 8 positives and 8 negatives → Entropy = 1.0 (maximum).

### Information Gain
Measures how much a feature split **reduces** entropy. Decision trees choose the feature with the **highest** Information Gain.

**`IG(S, A) = Entropy(S) - Σ (|Sv| / |S|) × Entropy(Sv)`**

- High IG: Split produces much purer child nodes — feature A is highly informative.
- IG = 0: Split provides no benefit.

### Gini Impurity
Measures the probability that a randomly chosen sample would be incorrectly classified if randomly labeled according to the node's class distribution.

**`Gini(S) = 1 - Σ p_i²`**

- Gini = 0: Perfect purity.
- Gini = 0.5 (binary): Maximum impurity — 50-50 split.
- Computationally faster than Entropy (no logarithm needed).

### ID3 vs CART
| | ID3 | CART |
|---|---|---|
| Criterion | Entropy + Information Gain | Gini Impurity (classification), MSE (regression) |
| Splits | Multi-way | Binary only |
| Features | Categorical only | Numerical + Categorical |
| Pruning | No native support | Cost-complexity pruning (ccp_alpha) |
| Used in | Historical | scikit-learn's DecisionTreeClassifier |


---
## Question 5: Linear SVM on 2-Feature Dataset

**Observations from the Visualization:**
- The black solid line (decision hyperplane) divides the feature space cleanly into two regions.
- The two dashed lines (margin boundaries at w·x+b = +1 and -1) are equidistant from the hyperplane. No training points fall inside the margin zone.
- Support vectors (circled in green) sit exactly ON the margin boundaries — these few points fully determine the hyperplane position.
- All other points are irrelevant to the model — moving them would not change the hyperplane.
- The margin zone appears empty — confirming this is a Hard Margin SVM with perfectly linearly separable data.


In [ ]:
# Question 5 – Linear SVM on 2-Feature Dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

# Generate linearly separable 2-feature dataset
X, y = make_classification(n_samples=50, n_features=2, n_informative=2,
                           n_redundant=0, n_clusters_per_class=1,
                           random_state=42)
y = np.where(y == 0, -1, 1)  # Convert labels to -1 and +1

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Linear SVM
svm = SVC(kernel='linear', C=1.0)
svm.fit(X_scaled, y)

# Extract hyperplane parameters
w = svm.coef_[0]
b = svm.intercept_[0]

# Create plot
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(X_scaled[y== 1, 0], X_scaled[y== 1, 1],
           color='blue', marker='o', s=80, label='Class +1', zorder=3)
ax.scatter(X_scaled[y==-1, 0], X_scaled[y==-1, 1],
           color='red',  marker='s', s=80, label='Class -1', zorder=3)

# Decision hyperplane and margins
x_vals = np.linspace(X_scaled[:,0].min()-1, X_scaled[:,0].max()+1, 300)
y_hyp  = -(w[0]*x_vals + b) / w[1]
y_pos  = -(w[0]*x_vals + b + 1) / w[1]
y_neg  = -(w[0]*x_vals + b - 1) / w[1]

ax.plot(x_vals, y_hyp, 'k-',  lw=2.5, label='Decision Hyperplane (w·x+b=0)')
ax.plot(x_vals, y_pos, 'b--', lw=1.5, label='+1 Margin boundary')
ax.plot(x_vals, y_neg, 'r--', lw=1.5, label='-1 Margin boundary')
ax.fill_between(x_vals, y_pos, y_neg, alpha=0.1, color='gray', label='Margin zone')

# Highlight support vectors
sv = svm.support_vectors_
ax.scatter(sv[:,0], sv[:,1], s=200, facecolors='none',
           edgecolors='green', linewidths=2.5, zorder=4, label='Support Vectors')

ax.set_xlabel('Feature 1 (scaled)'); ax.set_ylabel('Feature 2 (scaled)')
ax.set_title('Linear SVM: Hyperplane, Margins, and Support Vectors')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Print key stats
print(f'Number of support vectors: {len(sv)}')
print(f'Support vectors per class: {svm.n_support_}')
print(f'Weight vector w: {w}')
print(f'Bias b: {b:.4f}')
print(f'Margin width: {2 / np.linalg.norm(w):.4f}')


---
## Question 6: Gaussian Naive Bayes vs SVM on Iris Dataset

**Analysis:**
- **Gaussian NB** typically achieves ~93–96% accuracy on Iris. It performs well because the Iris classes are reasonably Gaussian-distributed in the feature space.
- **SVM (RBF kernel)** typically achieves ~97–100% accuracy on Iris, often outperforming GNB because it can model slight non-linearities in the feature boundaries between versicolor and virginica (the two overlapping classes).
- GNB is instantaneous to train and requires no feature scaling. SVM requires scaling and hyperparameter tuning but achieves higher accuracy.
- For larger or noisier datasets, SVM's superiority would likely be more pronounced.


In [ ]:
# Question 6 – Gaussian Naive Bayes vs SVM on Iris Dataset

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale for SVM
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Gaussian Naive Bayes ──
gnb = GaussianNB()
gnb.fit(X_train, y_train)       # GNB doesn't need scaling
y_pred_gnb = gnb.predict(X_test)

# ── SVM with RBF kernel ──
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_rbf.fit(X_train_s, y_train)
y_pred_svm = svm_rbf.predict(X_test_s)

# ── Results ──
print('=== Gaussian Naive Bayes ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_gnb)*100:.2f}%')
print(classification_report(y_test, y_pred_gnb, target_names=iris.target_names))

print('=== SVM (RBF Kernel) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_svm)*100:.2f}%')
print(classification_report(y_test, y_pred_svm, target_names=iris.target_names))


---
## Question 7: Telecom Churn Prediction — Algorithm Selection

### Recommended Algorithm: Decision Tree

Among SVM, Naive Bayes, and Decision Tree, the **Decision Tree** is recommended because:
- **Mixed data types:** Handles categorical (Contract Type, Payment Method) and numerical (Monthly Charges, Tenure) features natively. SVM requires encoding + scaling; NB requires careful variant selection.
- **Interpretability:** Stakeholders need to understand WHY a customer is predicted to churn (e.g., 'Month-to-month contract + high charges + low tenure → churn'). Decision Trees provide clear if-then rules.
- **Feature interactions:** Churn depends on feature combinations. Decision Trees capture interactions naturally; Naive Bayes' independence assumption misses these.
- **No feature scaling required:** Decision Trees are invariant to monotonic feature transformations.

### Preprocessing Steps
- **Handle missing values:** Median imputation for numerical; mode or 'Unknown' for categorical.
- **Encode categoricals:** Ordinal/Label Encoding for Decision Trees (they handle ordinal relationships).
- **Handle class imbalance:** Use `class_weight='balanced'` or SMOTE oversampling (churn is typically ~15–25%).
- **Feature engineering:** Create derived features like `charge_per_tenure`, `contract_duration_bucket`.
- **Stratified 80/20 split** to maintain churn ratio.

### Model Evaluation Strategy
- **Primary: Recall** — Minimize false negatives (missed churners = lost revenue). Target > 75%.
- **Secondary: Precision** — Avoid over-contacting non-churners (wasted retention costs).
- **F1-Score** — Harmonic mean; use as the tuning metric for hyperparameter optimization.
- **ROC-AUC** — Threshold-independent; target AUC > 0.80.
- **Confusion Matrix** — Explicitly examine TP, FN, FP for business validation.


---
## Question 8: SVM Analysis on Given Data Points

| F1 | F2 | Class |
|----|----|-------|
| 2  | 3  | +1    |
| 3  | 4  | +1    |
| 5  | 1  | -1    |
| 6  | 2  | -1    |

### Is the Data Linearly Separable?
**YES.** Class +1 points (2,3) and (3,4) are in the upper-left region; Class -1 points (5,1) and (6,2) are in the lower-right region. A straight line can clearly divide these two groups.

### Possible Separating Hyperplane
Approximate hyperplane based on centroids:
- Centroid of +1: (2.5, 3.5) | Centroid of -1: (5.5, 1.5)
- **Approximate hyperplane: `3F1 - 2F2 - 7 = 0`**

### Support Vectors
- **From Class +1:** Point **(3, 4)** — closest to the boundary.
- **From Class -1:** Point **(5, 1)** — closest to the boundary.
- Points (2,3) and (6,2) are farther away and are NOT support vectors.

### Effect of Changing the Margin
- **Increasing margin (lower C):** Hyperplane is forced further from both classes. Larger margin improves generalization but may misclassify some training points near the boundary.
- **Decreasing margin (higher C):** More training points classified correctly but model becomes noise-sensitive — overfitting risk.


In [ ]:
# Question 8 – SVM Analysis on Given Data Points

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import SVC

# Create dataframe
df = pd.DataFrame({
    'Feature_1': [2, 3, 5, 6],
    'Feature_2': [3, 4, 1, 2],
    'Class':     [1, 1,-1,-1]
})
print(df)

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(df[df.Class==1]['Feature_1'], df[df.Class==1]['Feature_2'],
            color='blue', marker='o', s=150, label='Class +1', zorder=5)
plt.scatter(df[df.Class==-1]['Feature_1'], df[df.Class==-1]['Feature_2'],
            color='red',  marker='s', s=150, label='Class -1', zorder=5)

# Label each point
for _, row in df.iterrows():
    plt.annotate(f'({int(row.Feature_1)},{int(row.Feature_2)})',
                 (row.Feature_1, row.Feature_2), textcoords='offset points',
                 xytext=(8, 5), fontsize=10)

# Train SVM and draw hyperplane
X = df[['Feature_1','Feature_2']].values
y = df['Class'].values
svm = SVC(kernel='linear', C=1.0)
svm.fit(X, y)
w = svm.coef_[0]; b = svm.intercept_[0]
x_vals = np.linspace(1, 7, 200)
plt.plot(x_vals, -(w[0]*x_vals + b)/w[1], 'k-', lw=2, label='Decision Hyperplane')
plt.plot(x_vals, -(w[0]*x_vals + b + 1)/w[1], 'b--', lw=1.5, label='+1 Margin')
plt.plot(x_vals, -(w[0]*x_vals + b - 1)/w[1], 'r--', lw=1.5, label='-1 Margin')

# Highlight support vectors
sv = svm.support_vectors_
plt.scatter(sv[:,0], sv[:,1], s=250, facecolors='none',
            edgecolors='green', linewidths=2.5, zorder=6, label='Support Vectors')

plt.xlabel('Feature 1'); plt.ylabel('Feature 2')
plt.title('SVM on Given Data Points: Hyperplane & Support Vectors')
plt.legend(); plt.grid(True, alpha=0.3)
plt.xlim(0, 8); plt.ylim(0, 6)
plt.tight_layout(); plt.show()

print(f'Support vectors: {svm.support_vectors_}')
print(f'Margin width: {2 / np.linalg.norm(w):.4f}')


---
## Question 9: Algorithm Selection for Three Datasets

### Dataset A: 500 rows, 5 numerical features, clearly separable classes → **SVM**
- 500 rows with 5 features makes SVM's training complexity manageable.
- Clearly separable classes mean SVM's maximum-margin approach will find an excellent boundary.
- SVM generalizes better than Decision Trees on small datasets due to the margin maximization principle.
- Naive Bayes' independence assumption may not hold for 5 correlated numerical features.

### Dataset B: 1 million rows, 500 text features, multi-class → **Naive Bayes (Multinomial NB)**
- Multinomial NB is the gold standard for text classification — extremely fast `O(n×d)` training.
- Handles 1 million rows in seconds; scales linearly with dataset size and feature count.
- Works naturally with sparse text features; handles multi-class natively.
- SVM's `O(n²–n³)` complexity makes training on 1M rows impractical without extensive engineering.

### Dataset C: 20,000 rows, categorical + numerical features, explainability required → **Decision Tree**
- Critical requirement is **explainability** — business users need human-readable if-then rules.
- Handles mixed data types without encoding or scaling (CART algorithm).
- 20,000 rows is the ideal size for a single decision tree.
- Feature importance scores show which variables drive predictions.
- SVM is a black box; Naive Bayes' independence assumption is violated with mixed business data.


---
## Question 10: Decision Tree Overfitting Analysis

### Is the Model Overfitting or Underfitting?
The model is **SEVERELY OVERFITTING**. Training accuracy of 99.8% with Test accuracy of 68% represents a generalization gap of **31.8 percentage points** — a classic and extreme overfitting signature.

### Signs That Indicate Overfitting
- **Massive training-test gap:** 99.8% vs 68% — a gap > 3–5% indicates overfitting.
- **Near-perfect training accuracy:** The tree has memorized the training set including noise and irrelevant patterns.
- Unconstrained trees grow until each leaf contains a single sample — achieving ~100% training accuracy but poor generalization.

### Four Methods to Improve Generalization
1. **Limit Tree Depth (`max_depth`):** Set `max_depth=5–10` — forces the model to learn broader generalizable patterns instead of memorizing specific training examples.
2. **Increase `min_samples_split`:** Set to 20–50 — requires each internal node to have sufficient samples before splitting, preventing splits on tiny noise-capturing subsets.
3. **Post-pruning (`ccp_alpha`):** Grow the full tree, then prune branches that don't justify their complexity. Use cross-validation to find the optimal alpha.
4. **Use Random Forest / Ensemble Methods:** Train multiple trees on bootstrapped subsets with random feature subsets. Averaging their predictions dramatically reduces overfitting through variance reduction.

### Role of Key Hyperparameters
- **`max_depth`:** Limits levels the tree can grow. Too small → underfitting; too large → overfitting. Tune via cross-validation.
- **`min_samples_split`:** Default is 2 (splits on any 2 samples). Increasing to 20–50 prevents noise-capturing splits.
- **`min_samples_leaf`:** Minimum samples required in any leaf node. Setting to 10–20 prevents tiny leaf nodes that memorize 1–3 training examples.
- **`ccp_alpha` (pruning):** Controls the complexity cost per branch. High alpha → simpler tree (higher bias, lower variance). Find optimal alpha by plotting train/validation accuracy vs alpha.
